# Дообучение ruBert-base на DaNetQA (Russian SuperGLUE) и Telegram-бот

**Задание**: взять одну задачу Russian SuperGLUE, зафайнтюнить модель с HuggingFace, сделать сабмит на лидерборд и встроить модель в телеграм-бота.

**Задача DaNetQA** — бинарный вопросно-ответный датасет (да/нет): дан отрывок (passage) и вопрос (question); ответ должен следовать только из отрывка. Формулировка близка к NLI: пара текстов классифицируется в 2 класса. Метрика — Accuracy. Человеческий бейзлайн — 0.915.

In [1]:
# version banner
import sys
import torch, transformers, datasets, sklearn
print('python', sys.version.split()[0])
print('torch', torch.__version__, '| mps:', torch.backends.mps.is_available())
print('transformers', transformers.__version__, '| datasets', datasets.__version__,
      '| sklearn', sklearn.__version__)
SEED = 42

python 3.12.13
torch 2.14.1 | mps: True
transformers 4.57.6 | datasets 5.0.1 | sklearn 1.9.1


## 1. Данные

Скачиваем официальный архив DaNetQA с russiansuperglue.com (скрипт идемпотентен) и смотрим на размеры, баланс классов и пример.

In [2]:
import json
from pathlib import Path
import pandas as pd

if not Path('data/DaNetQA/train.jsonl').exists():
    import subprocess; subprocess.run(['.venv/bin/python', 'scripts/download_data.py'], check=True)

def read_jsonl(p):
    return [json.loads(l) for l in open(p, encoding='utf-8')]

rows = {s: read_jsonl(f'data/DaNetQA/{s}.jsonl') for s in ['train', 'val', 'test']}
for s, r in rows.items():
    labels = [x.get('label') for x in r]
    print(s, len(r), '| поля:', sorted(r[0]), '| label true:', sum(1 for l in labels if l))

ex = rows['train'][0]
print('\nПример:')
print('passage:', ex['passage'][:200], '...')
print('question:', ex['question'])
print('label:', ex['label'])

train 1749 | поля: ['idx', 'label', 'passage', 'question'] | label true: 1061
val 821 | поля: ['idx', 'label', 'passage', 'question'] | label true: 412
test 805 | поля: ['idx', 'passage', 'question'] | label true: 0

Пример:
passage: «Вы́ставочный центр» — станция Московского монорельса. Расположена между станциями «Улица Академика Королёва» и «Улица Сергея Эйзенштейна». Находится на территории Останкинского района Северо-Восточно ...
question: Вднх - это выставочный центр?
label: True


## 2. Токенизация

Пара текстов подаётся как `(passage, question)` — ровно так же, как потом в боте (`inference.py`). Обрезка — `truncation='only_first'`: режется хвост passage, вопрос сохраняется целиком. `MAX_LEN=256` — проверим по статистике длин.

In [3]:
from transformers import AutoTokenizer

MODEL_NAME = 'ai-forever/ruBert-base'
MAX_LEN = 256
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

lens = [len(tokenizer(x['passage'], x['question'])['input_ids']) for x in rows['train'] + rows['val']]
s = pd.Series(lens)
print(s.describe().round(1))
for n in [192, 256, 320, 384, 512]:
    print(f'покрытие max_len={n}: {(s <= n).mean():.1%}')

count    2570.0
mean      158.6
std        56.3
min        64.0
25%       123.0
50%       146.0
75%       178.0
max       751.0
dtype: float64
покрытие max_len=192: 81.3%
покрытие max_len=256: 94.7%
покрытие max_len=320: 98.2%
покрытие max_len=384: 99.4%
покрытие max_len=512: 99.8%


In [4]:
import datasets as hfds

# у test-сплита метки скрыты — заглушка 0 (accuracy по ним не осмыслен)
def to_ds(split):
    items = [{'label': int(bool(x['label'])) if 'label' in x else 0}
             for x in rows[split]]
    texts = [(x['passage'], x['question']) for x in rows[split]]
    ds = hfds.Dataset.from_list(items)
    return ds.map(lambda _, i: tokenizer(texts[i][0], texts[i][1],
                    truncation='only_first', max_length=MAX_LEN),
                   with_indices=True)

ds = {s: to_ds(s) for s in ['train', 'val', 'test']}
ds['train'][0]['label']

Map:   0%|          | 0/1749 [00:00<?, ? examples/s]

Map:   0%|          | 0/821 [00:00<?, ? examples/s]

Map:   0%|          | 0/805 [00:00<?, ? examples/s]

1

## 3. Дообучение ruBert-base

`AutoModelForSequenceClassification` (num_labels=2), HF Trainer на устройстве mps (Apple M1 Max): lr 2e-5, batch 16, 3 эпохи, warmup 10%, eval по эпохам, лучший чекпойнт по accuracy на val.

In [5]:
import numpy as np
from sklearn.metrics import accuracy_score
from transformers import AutoModelForSequenceClassification, Trainer, TrainingArguments

model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=2)
model.config.id2label = {0: 'false', 1: 'true'}
model.config.label2id = {'false': 0, 'true': 1}

def compute_metrics(p):
    return {'accuracy': accuracy_score(p.label_ids, np.argmax(p.predictions, axis=-1))}

args = TrainingArguments(
    output_dir='outputs/train',
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=64,
    num_train_epochs=3,
    weight_decay=0.01,
    warmup_ratio=0.1,
    eval_strategy='epoch',
    save_strategy='epoch',
    load_best_model_at_end=True,
    metric_for_best_model='accuracy',
    save_total_limit=1,
    seed=SEED,
    report_to=[],
    dataloader_pin_memory=False,  # без ворнингов на mps
)
trainer = Trainer(model=model, args=args,
                  train_dataset=ds['train'], eval_dataset=ds['val'],
                  processing_class=tokenizer, compute_metrics=compute_metrics)
trainer.train()

pytorch_model.bin:   0%|          | 0.00/716M [00:00<?, ?B/s]

Some weights of BertForSequenceClassification were not initialized from the model checkpoint at ai-forever/ruBert-base and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


model.safetensors:   0%|          | 0.00/716M [00:00<?, ?B/s]

Epoch,Training Loss,Validation Loss,Accuracy
1,No log,0.660530,0.621194
2,No log,0.804780,0.612667
3,No log,0.951385,0.618758


TrainOutput(global_step=330, training_loss=0.40337242357658615, metrics={'train_runtime': 347.8772, 'train_samples_per_second': 15.083, 'train_steps_per_second': 0.949, 'total_flos': 660490457269620.0, 'train_loss': 0.40337242357658615, 'epoch': 3.0})

## 4. Оценка на val

Сравниваем с тривиальными опорными точками: константный бейзлайн (всегда «да») и человеческий бейзлайн 0.915.

In [6]:
from sklearn.metrics import classification_report

eval_out = trainer.evaluate()
val_acc = eval_out['eval_accuracy']
majority = sum(bool(x['label']) for x in rows['val']) / len(rows['val'])
print(f'val accuracy: {val_acc:.3f}')
print(f'бейзлайн «всегда да»: {majority:.3f}')
print('человеческий бейзлайн: 0.915')

pred = np.argmax(trainer.predict(ds['val']).predictions, axis=-1)
gold = [int(bool(x['label'])) for x in rows['val']]
print(classification_report(gold, pred, target_names=['false', 'true'], digits=3))

val accuracy: 0.621
бейзлайн «всегда да»: 0.502
человеческий бейзлайн: 0.915


              precision    recall  f1-score   support

       false      0.702     0.416     0.522       409
        true      0.587     0.825     0.686       412

    accuracy                          0.621       821
   macro avg      0.645     0.620     0.604       821
weighted avg      0.645     0.621     0.605       821



## 5. Сохранение модели для бота

Чекпойнт и метрика — в `outputs/danetqa-rubert/`; бот (`bot.py`) и smoke-тест (`scripts/smoke_test.py`) грузят их через `inference.py`.

In [7]:
import json

OUT = Path('outputs/danetqa-rubert')
OUT.mkdir(parents=True, exist_ok=True)
trainer.save_model(OUT)
tokenizer.save_pretrained(OUT)
json.dump({'val_accuracy': val_acc, 'max_len': MAX_LEN, 'seed': SEED},
          open(OUT / 'metrics.json', 'w'), ensure_ascii=False, indent=2)
print('сохранено:', OUT, '| val_accuracy =', round(val_acc, 4))

сохранено: outputs/danetqa-rubert | val_accuracy = 0.6212


## 6. Предсказания на test и сабмит

Формат — как в `sample_submission.zip` официального репозитория: JSONL `{"idx": <int>, "label": "true"|"false"}`. Файл загружается в личном кабинете https://russiansuperglue.com/leaderboard/2.

In [8]:
test_pred = np.argmax(trainer.predict(ds['test']).predictions, axis=-1)
sub = Path('outputs/submission'); sub.mkdir(parents=True, exist_ok=True)
with open(sub / 'DaNetQA.jsonl', 'w', encoding='utf-8') as f:
    for x, p in zip(rows['test'], test_pred):
        f.write(json.dumps({'idx': x['idx'], 'label': 'true' if p == 1 else 'false'},
                            ensure_ascii=False) + '\n')
print('строк в сабмите:', sum(1 for _ in open(sub / 'DaNetQA.jsonl')))
print('классы:', pd.Series(test_pred).map({0: 'false', 1: 'true'}).value_counts().to_dict())
print(open(sub / 'DaNetQA.jsonl').readline().strip())

строк в сабмите: 805
классы: {'true': 490, 'false': 315}
{"idx": 0, "label": "true"}


## 7. Демо: как отвечает модель (тот же путь, что в боте)

Три примера из val — passage/question/предсказание/эталон.

In [9]:
import sys; sys.path.insert(0, '.')
from inference import DanetqaModel, format_answer

m = DanetqaModel(trainer.model, tokenizer, max_len=MAX_LEN)
for x in rows['val'][:3]:
    p, _ = m.predict(x['passage'], x['question'])
    print(f"Q: {x['question']}")
    print(f"  ответ модели: {format_answer(p)} | эталон: {'Да' if x['label'] else 'Нет'}")

Q: Есть ли вода на марсе?
  ответ модели: Да (уверенность 77%) | эталон: Да
Q: Состоит ли англия в евросоюзе?
  ответ модели: Нет (уверенность 71%) | эталон: Нет
Q: Действительно ли в ссср не было адвокатов?
  ответ модели: Да (уверенность 84%) | эталон: Нет


## 8. Результат сабмита на лидерборд

Скриншот страницы сабмита с russiansuperglue.com/leaderboard/2 (скрины лежат в `assets/`, наличие проверяет `scripts/fill_screenshots.py`):

![leaderboard](assets/leaderboard.png)

## 9. Telegram-бот

Бот на aiogram 3 (`bot.py`): `/start` печатает описание задачи DaNetQA, далее FSM-диалог «отрывок → вопрос → ответ Да/Нет с уверенностью». Запуск: `.venv/bin/python bot.py` (токен в `.env`). Скриншоты работы:

![бот 1](assets/bot_intro.png)

![бот 2](assets/bot_dialog.png)